In [0]:
from datetime import datetime
from pyspark.sql.functions import lit

#User pass date
try:
	arrival_date=dbutils.widgets.get("arrival_date")
except Exception:
	arrival_date=datetime.now().strftime("%Y-%m-%d")

#User pass catalog
try:
	catalog=dbutils.widgets.get("catalog")
except Exception:
	catalog="travel_bookings"

#User pass schema
try:
	schema=dbutils.widgets.get("schema")
except Exception:
	schema="default"

#User pass volume 
try:
	base_volume=dbutils.widgets.get("volume")
except Exception:
	base_volume=f"/Volumes/{catalog}/{schema}/data"

In [0]:
#Read customer file and write into bronze layer table
customer_path=f"{base_volume}/customer_data/customers_{arrival_date}.csv"
df=spark.read.csv(customer_path, header=True, inferSchema=True, escape='"', multiLine=True)

#Normalize SCD2 dates 
new_df=df.withColumn("is_current", lit(True)).withColumn("business_date", lit(arrival_date))
spark.sql(f"create schema if not exists {catalog}.bronze")
new_df.write.format("delta").mode("append").saveAsTable(f"{catalog}.bronze.customer_inc")